In [2]:
# 04_Featuring.py
import pandas as pd
import numpy as np
import os

INPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\normalized"
OUTPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\featuring"
METRIC_PATH = os.path.join(OUTPUT_PATH, "metricas_generacion")
os.makedirs(METRIC_PATH, exist_ok=True)

def add_features(df):
    df_extended = df.copy().astype(np.float32)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = [col for col in numeric_cols if df[col].nunique() > 2]

    detalle_columnas = []

    for col in numeric_cols:
        try:
            col_data = df[col].replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)

            df_extended[f"{col}_log"] = np.log1p(np.abs(col_data))
            detalle_columnas.append((col, f"{col}_log", "log"))

            df_extended[f"{col}_sqrt"] = np.sqrt(np.abs(col_data))
            detalle_columnas.append((col, f"{col}_sqrt", "sqrt"))

            df_extended[f"{col}_squared"] = col_data ** 2
            detalle_columnas.append((col, f"{col}_squared", "squared"))

        except Exception as e:
            print(f"⚠️ Error en columna {col}: {e}")

    return df_extended, pd.DataFrame(detalle_columnas, columns=["original", "generada", "tipo"])

# Procesar cada subcarpeta
for norm_name in os.listdir(INPUT_PATH):
    norm_dir = os.path.join(INPUT_PATH, norm_name)
    out_dir = os.path.join(OUTPUT_PATH, norm_name)
    os.makedirs(out_dir, exist_ok=True)

    try:
        X_train = pd.read_parquet(f"{norm_dir}/X_train_{norm_name}.parquet")
        X_test = pd.read_parquet(f"{norm_dir}/X_test_{norm_name}.parquet")
        y_train = pd.read_parquet(f"{norm_dir}/y_train_{norm_name}.parquet")
        y_test = pd.read_parquet(f"{norm_dir}/y_test_{norm_name}.parquet")
    except Exception as e:
        print(f"❌ Error leyendo archivos en {norm_name}: {e}")
        continue

    # Guardar versión original sin features
    X_train.to_parquet(f"{out_dir}/X_train_{norm_name}_ORIGINAL.parquet")
    X_test.to_parquet(f"{out_dir}/X_test_{norm_name}_ORIGINAL.parquet")
    y_train.to_parquet(f"{out_dir}/y_train_{norm_name}_ORIGINAL.parquet")
    y_test.to_parquet(f"{out_dir}/y_test_{norm_name}_ORIGINAL.parquet")

    # Aplicar feature engineering y obtener detalle
    X_train_fe, df_detalle = add_features(X_train)
    X_test_fe, _ = add_features(X_test)

    # Guardar datasets con features
    X_train_fe.to_parquet(f"{out_dir}/X_train_{norm_name}_FE.parquet")
    X_test_fe.to_parquet(f"{out_dir}/X_test_{norm_name}_FE.parquet")
    y_train.to_parquet(f"{out_dir}/y_train_{norm_name}_FE.parquet")
    y_test.to_parquet(f"{out_dir}/y_test_{norm_name}_FE.parquet")

    # Guardar detalle de columnas generadas
    detalle_file = os.path.join(METRIC_PATH, f"detalle_columnas_{norm_name}.csv")
    df_detalle.to_csv(detalle_file, index=False)

    # Métricas en consola
    resumen = df_detalle["tipo"].value_counts()
    print(f"\n📊 {norm_name} - Ingeniería aplicada")
    print(f"📌 Columnas originales: {X_train.shape[1]}")
    print(f"🧪 Nuevas columnas generadas: {df_detalle.shape[0]}")
    print(f"🧮 Total final de columnas: {X_train_fe.shape[1]}")
    print("🔍 Detalle por tipo de transformación:")
    for tipo, count in resumen.items():
        print(f"   ➤ {tipo}: {count}")
    print(f"📤 CSV guardado en: {detalle_file}")



C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_16980\601011091.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_extended[f"{col}_log"] = np.log1p(np.abs(col_data))
C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_16980\601011091.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_extended[f"{col}_sqrt"] = np.sqrt(np.abs(col_data))
C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_16980\601011091.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which h


📊 MaxAbs - Ingeniería aplicada
📌 Columnas originales: 539
🧪 Nuevas columnas generadas: 441
🧮 Total final de columnas: 980
🔍 Detalle por tipo de transformación:
   ➤ log: 147
   ➤ sqrt: 147
   ➤ squared: 147
📤 CSV guardado en: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\featuring\metricas_generacion\detalle_columnas_MaxAbs.csv


C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_16980\601011091.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_extended[f"{col}_log"] = np.log1p(np.abs(col_data))
C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_16980\601011091.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_extended[f"{col}_sqrt"] = np.sqrt(np.abs(col_data))
C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_16980\601011091.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which h


📊 MinMax - Ingeniería aplicada
📌 Columnas originales: 539
🧪 Nuevas columnas generadas: 441
🧮 Total final de columnas: 980
🔍 Detalle por tipo de transformación:
   ➤ log: 147
   ➤ sqrt: 147
   ➤ squared: 147
📤 CSV guardado en: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\featuring\metricas_generacion\detalle_columnas_MinMax.csv


C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_16980\601011091.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_extended[f"{col}_squared"] = col_data ** 2
C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_16980\601011091.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_extended[f"{col}_log"] = np.log1p(np.abs(col_data))
C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_16980\601011091.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor p


📊 NoNorm - Ingeniería aplicada
📌 Columnas originales: 539
🧪 Nuevas columnas generadas: 441
🧮 Total final de columnas: 980
🔍 Detalle por tipo de transformación:
   ➤ log: 147
   ➤ sqrt: 147
   ➤ squared: 147
📤 CSV guardado en: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\featuring\metricas_generacion\detalle_columnas_NoNorm.csv
❌ Error leyendo archivos en resumen_columnas_normalizacion.csv: [Errno 2] No such file or directory: 'C:\\Users\\Gonzalo\\Documents\\GitHub\\TesisAustral\\Archivos_tesis\\intermediate\\normalized\\resumen_columnas_normalizacion.csv/X_train_resumen_columnas_normalizacion.csv.parquet'


C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_16980\601011091.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_extended[f"{col}_log"] = np.log1p(np.abs(col_data))
C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_16980\601011091.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_extended[f"{col}_sqrt"] = np.sqrt(np.abs(col_data))
C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_16980\601011091.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which h


📊 Robust - Ingeniería aplicada
📌 Columnas originales: 539
🧪 Nuevas columnas generadas: 441
🧮 Total final de columnas: 980
🔍 Detalle por tipo de transformación:
   ➤ log: 147
   ➤ sqrt: 147
   ➤ squared: 147
📤 CSV guardado en: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\featuring\metricas_generacion\detalle_columnas_Robust.csv


C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_16980\601011091.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_extended[f"{col}_log"] = np.log1p(np.abs(col_data))
C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_16980\601011091.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_extended[f"{col}_sqrt"] = np.sqrt(np.abs(col_data))
C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_16980\601011091.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which h


📊 Standard - Ingeniería aplicada
📌 Columnas originales: 539
🧪 Nuevas columnas generadas: 441
🧮 Total final de columnas: 980
🔍 Detalle por tipo de transformación:
   ➤ log: 147
   ➤ sqrt: 147
   ➤ squared: 147
📤 CSV guardado en: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\featuring\metricas_generacion\detalle_columnas_Standard.csv
